In [5]:
import networkx as nx
import math
import re

# 1. TSP 파일 파싱
nodes = {}
# with open("a280.tsp", encoding="utf-8") as f:
with open("xql662.tsp", encoding="utf-8") as f:
# with open("kz9976.tsp", encoding="utf-8") as f:
    start = False
    for line in f:
        if "NODE_COORD_SECTION" in line:
            start = True
            continue
        if start:
            if "EOF" in line or not line.strip():
                break
            parts = re.split(r"\s+", line.strip())
            if len(parts) >= 3:
                node_id, x, y = int(parts[0]), float(parts[1]), float(parts[2])
                nodes[node_id] = (x, y)

# 좌표 출력 함수 정의
def format_coords(u, v):
    return f'{{{{{u}, {int(nodes[u][0])}, {int(nodes[u][1])}}}, {{{v}, {int(nodes[v][0])}, {int(nodes[v][1])}}}}}'


def format_node(n):
    return f'{{{n}, {int(nodes[n][0])}, {int(nodes[n][1])}}}'



# 2. 완전 그래프 생성 (유클리드 거리)
G = nx.Graph()
for i, (x1, y1) in nodes.items():
    for j, (x2, y2) in nodes.items():
        if i < j:
            dist = math.hypot(x1 - x2, y1 - y2)
            G.add_edge(i, j, weight=dist)

# 3. MST 계산
mst = nx.minimum_spanning_tree(G)
mst_edges = list(mst.edges())
mst_total_weight = sum(G[u][v]['weight'] for u, v in mst.edges())
mst_edges_str = ', '.join(format_coords(u, v) for u, v in mst_edges)
print("\n[MST] 엣지:", mst_edges_str)
print("[MST] 총 거리:", mst_total_weight)

# 4. MST에서 홀수 차수 노드 추출
odd_degree_nodes = [v for v, d in mst.degree() if d % 2 == 1]
odd_nodes_str = ', '.join(format_node(v) for v in odd_degree_nodes)
print("\n[홀수 차수 노드] 노드:", odd_nodes_str)
print("[홀수 차수 노드] 개수:", len(odd_degree_nodes))


# 5. 홀수 차수 노드끼리 최소 가중치 완전 매칭
odd_subgraph = G.subgraph(odd_degree_nodes)
min_weight_matching = nx.algorithms.matching.min_weight_matching(odd_subgraph, weight='weight')
matching_edges = list(min_weight_matching)
matching_total_weight = sum(G[u][v]['weight'] for u, v in matching_edges)
matching_edges_str = ', '.join(format_coords(u, v) for u, v in matching_edges)
print("\n[최소 가중치 매칭] 엣지:", matching_edges_str)
print("[최소 가중치 매칭] 총 거리:", matching_total_weight)

# Perfect Matching 여부 확인
is_perfect = nx.algorithms.matching.is_perfect_matching(odd_subgraph, min_weight_matching)
print("[매칭이 Perfect Matching인가?]", is_perfect)

# 6. MST와 매칭을 합쳐 멀티그래프 구성
multigraph = nx.MultiGraph(mst)
multigraph.add_edges_from(matching_edges)

# 7. 오일러 경로 생성
eulerian_circuit = list(nx.eulerian_circuit(multigraph))
eulerian_edges_str = ', '.join(format_coords(u, v) for u, v in eulerian_circuit)
eulerian_total_weight = sum(G[u][v]['weight'] for u, v in eulerian_circuit)
print("\n[오일러 경로] 엣지:", eulerian_edges_str)
print("[오일러 경로] 총 거리:", eulerian_total_weight)

# 8. 오일러 경로를 shortcut하여 TSP 경로 생성
visited = set()
tsp_path = []
for u, v in eulerian_circuit:
    if u not in visited:
        tsp_path.append(u)
        visited.add(u)
    if v not in visited:
        tsp_path.append(v)
        visited.add(v)
# 시작점으로 돌아오기
tsp_path.append(tsp_path[0])
tsp_path_str = ', '.join(format_node(n) for n in tsp_path)
tsp_total_weight = sum(G[tsp_path[i]][tsp_path[i+1]]['weight'] for i in range(len(tsp_path)-1))
print("\n[TSP 경로] 노드:", tsp_path_str)
print("[TSP 경로] 총 거리:", tsp_total_weight)


[MST] 엣지: {{1, 0, 20}, {8, 2, 16}}, {{1, 0, 20}, {9, 2, 25}}, {{2, 0, 39}, {15, 2, 39}}, {{3, 0, 45}, {17, 2, 44}}, {{3, 0, 45}, {18, 2, 47}}, {{4, 0, 58}, {23, 2, 59}}, {{4, 0, 58}, {22, 2, 56}}, {{5, 0, 64}, {25, 2, 64}}, {{6, 0, 77}, {31, 2, 77}}, {{7, 2, 11}, {33, 2, 9}}, {{7, 2, 11}, {8, 2, 16}}, {{8, 2, 16}, {42, 10, 15}}, {{9, 2, 25}, {10, 2, 31}}, {{10, 2, 31}, {11, 2, 32}}, {{11, 2, 32}, {12, 2, 33}}, {{12, 2, 33}, {13, 2, 35}}, {{13, 2, 35}, {14, 2, 38}}, {{13, 2, 35}, {37, 5, 36}}, {{14, 2, 38}, {15, 2, 39}}, {{15, 2, 39}, {34, 3, 41}}, {{16, 2, 43}, {17, 2, 44}}, {{16, 2, 43}, {34, 3, 41}}, {{18, 2, 47}, {19, 2, 48}}, {{19, 2, 48}, {20, 2, 49}}, {{20, 2, 49}, {21, 2, 54}}, {{21, 2, 54}, {22, 2, 56}}, {{23, 2, 59}, {24, 2, 60}}, {{24, 2, 60}, {35, 3, 62}}, {{25, 2, 64}, {26, 2, 65}}, {{25, 2, 64}, {35, 3, 62}}, {{26, 2, 65}, {27, 2, 68}}, {{27, 2, 68}, {28, 2, 69}}, {{28, 2, 69}, {29, 2, 70}}, {{29, 2, 70}, {30, 2, 75}}, {{30, 2, 75}, {31, 2, 77}}, {{31, 2, 77}, {32, 2, 82}

In [9]:
import math

# 정답 파악하기

# TSP 파일에서 좌표 읽기
def read_tsp(filepath):
    coords = {}
    with open(filepath, 'r') as f:
        node_section = False
        for line in f:
            line = line.strip()
            if line == "NODE_COORD_SECTION":
                node_section = True
                continue
            if line == "EOF":
                break
            if node_section:
                parts = line.split()
                if len(parts) >= 3:
                    node_id = int(parts[0])
                    x, y = float(parts[1]), float(parts[2])
                    coords[node_id] = (x, y)
    return coords

# TOUR 파일에서 경로 읽기
def read_tour(filepath):
    tour = []
    with open(filepath, 'r') as f:
        tour_section = False
        for line in f:
            line = line.strip()
            if line == "TOUR_SECTION":
                tour_section = True
                continue
            if line == "-1" or line == "EOF":
                break
            if tour_section:
                tour.append(int(line))
    return tour

# 두 점 사이의 유클리드 거리 계산
# def euclidean(p1, p2):
#     return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

# 두 점 사이의 유클리드 거리 계산 (TSPLIB 규칙 적용)
def euclidean(p1, p2):
    return int(math.hypot(p1[0] - p2[0], p1[1] - p2[1]) + 0.5)



# 전체 투어 길이 계산
def calculate_tour_length(coords, tour):
    total_dist = 0.0
    n = len(tour)
    for i in range(n):
        a = tour[i]
        b = tour[(i + 1) % n]  # 순환 경로 고려
        total_dist += euclidean(coords[a], coords[b])
    return total_dist

# 파일 경로 설정 (필요 시 경로 수정)
# tsp_path = "a280.tsp"
# tour_path = "a280.opt.tour"
# tsp_path = "xql662.tsp"
# tour_path = "xql662.tour"
tsp_path = "kz9976.tsp"
tour_path = "kz9976.tour"

# 좌표 및 경로 읽기
coords = read_tsp(tsp_path)
tour = read_tour(tour_path)

# 총 경로 길이 계산
total_length = calculate_tour_length(coords, tour)

# 결과 출력
print(f"총 경로 길이: {total_length:.2f}")


총 경로 길이: 1061882.00
